In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
import io

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

In [5]:
def get_html(url):
    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()
    return resp.text

In [7]:
def get_stats_table(year, stat_type):

    if stat_type == "per_game":
        table_id = "per_game_stats"
    else:
        table_id = "advanced"
    
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_{stat_type}.html"
    
    soup = BeautifulSoup(get_html(url), "html.parser")
    table = soup.find("table", {"id": table_id})

    # turn the html table into a dataframe
    # (wrap the html string in io.StringIO, otherwise pandas gives a warning)
    df = pd.read_html(io.StringIO(str(table)))[0]

    # the table repeats the header row every 20 players, remove those
    df = df[df["Player"] != "Player"]

    # get player_id
    player_id_list = []
    rows = table.find("tbody").find_all("tr")
    
    for row in rows:
        if "thead" in row.get("class", []):
            continue
        player_cell = row.find("td", {"data-stat": "name_display"})
        if player_cell is not None:
            player_id_list.append(player_cell.get("data-append-csv"))
        else:
            player_id_list.append(None)

    df["player_id"] = player_id_list
    df["season"] = str(year - 1) + "-" + str(year)[2:]

    return df

In [9]:
per_game_tables = []
advanced_tables = []

for year in range(2020, 2025):
    print("Season:", year)

    per_game_df = get_stats_table(year, "per_game")
    per_game_tables.append(per_game_df)
    time.sleep(4)

    advanced_df = get_stats_table(year, "advanced")
    advanced_tables.append(advanced_df)
    time.sleep(4)

final_per_game = pd.concat(per_game_tables)
final_advanced = pd.concat(advanced_tables)

final_per_game.to_csv("player_per_game_raw.csv", index=False)
final_advanced.to_csv("player_advanced_raw.csv", index=False)

Season: 2020
Season: 2021
Season: 2022
Season: 2023
Season: 2024
